# DeepConvLSTM Replication Notebook (Baseline + PTQ + QAT)

This notebook replicates the DeepConvLSTM pipeline aligned with the **Efficient human activity recognition on edge devices using DeepConv LSTM architectures** paper using this repo's implementation.

## Scope
- Baseline, PTQ, and QAT experiments.
- Both split protocols: `random_stratified` and `user_holdout`.
- Two run modes:
  - `quick`: short sanity execution.
  - `full`: full-fidelity replication settings.

This notebook reuses `src/` modules directly to avoid logic drift.


In [ ]:
from pathlib import Path
import sys
import os
import json
import copy
import importlib.util

# Toggle this before running the cell if your kernel crashes on GPU.
USE_GPU = True
if not USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# --- TensorFlow GPU/XLA runtime guardrails (must run before importing project modules) ---
# In this conda env, NVIDIA CUDA libs and ptxas may live under site-packages/nvidia/*.
py_major, py_minor = sys.version_info[:2]
nvidia_root = Path(sys.prefix) / f"lib/python{py_major}.{py_minor}/site-packages/nvidia"
cuda_nvcc_root = nvidia_root / "cuda_nvcc"
ptxas_bin_dir = cuda_nvcc_root / "bin"
ptxas_path = ptxas_bin_dir / "ptxas"

if USE_GPU and nvidia_root.exists():
    # Ensure CUDA shared libraries (including libnvrtc) are discoverable.
    lib_dirs = [p for p in sorted(nvidia_root.glob("*/lib")) if p.is_dir()]
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    ld_entries = existing_ld.split(":") if existing_ld else []
    for lib_dir in reversed(lib_dirs):
        s = str(lib_dir)
        if s not in ld_entries:
            ld_entries.insert(0, s)
    os.environ["LD_LIBRARY_PATH"] = ":".join(ld_entries)

    # Ensure ptxas is on PATH for XLA CUDA compilation.
    if ptxas_path.exists():
        path_entries = os.environ.get("PATH", "").split(":") if os.environ.get("PATH") else []
        if str(ptxas_bin_dir) not in path_entries:
            os.environ["PATH"] = f"{ptxas_bin_dir}:{os.environ.get('PATH', '')}".strip(":")

        # Hint XLA where CUDA toolchain data lives.
        xla_flag = f"--xla_gpu_cuda_data_dir={cuda_nvcc_root}"
        existing_xla = os.environ.get("XLA_FLAGS", "")
        if xla_flag not in existing_xla:
            os.environ["XLA_FLAGS"] = (existing_xla + " " + xla_flag).strip()

# # Avoid greedy VRAM reservation in notebook sessions.
# os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
    "yaml": "PyYAML",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow",
    "tensorflow_model_optimization": "tensorflow-model-optimization",
}
missing = [f"{mod} (pip package: {pkg})" for mod, pkg in required_modules.items() if importlib.util.find_spec(mod) is None]
if missing:
    missing_text = "\n- ".join(missing)
    raise ModuleNotFoundError(
        "Missing required notebook dependencies:\n- "
        + missing_text
        + "\n\nActivate tinymlproj and install dependencies:\n"
        + "conda activate tinymlproj && conda env update -n tinymlproj -f environment.yml --prune"
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import tensorflow as tf

# Configure TF GPU memory growth early (before any real TF GPU allocations happen)
if USE_GPU:
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception as e:
            print(f"[WARN] Could not set memory growth for {g}: {e}")

print("TF version:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# Ensure repo root is importable when launched from different working directories.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_yaml
from src.utils.runtime import check_tensorflow_runtime
from src.utils.repro import load_json, set_global_seed
from src.utils.artifacts import baseline_ckpt_path, history_path, split_npz_path

from src.data.load_wisdm import load_wisdm_dataframe
from src.data.preprocess_zhou2025 import preprocess_zhou2025
from src.data.build_dataset import build_dataset_for_protocol

from src.train.train_baseline import train_baseline_for_protocol
from src.eval.eval_baseline import evaluate_baseline_for_protocol

from src.quant.ptq_full_int8 import quantize_ptq_for_protocol
from src.quant.qat_train import qat_for_protocol
from src.eval.eval_tflite import evaluate_tflite

nvrtc_candidates = []
if nvidia_root.exists():
    for lib_dir in sorted(nvidia_root.glob("*/lib")):
        nvrtc_candidates.extend(sorted(lib_dir.glob("libnvrtc.so*")))

if USE_GPU and not nvrtc_candidates:
    raise RuntimeError(
        "GPU mode requested but libnvrtc.so was not found in tinymlproj. "
        "Install/refresh env with: conda activate tinymlproj && "
        "conda env update -n tinymlproj -f environment.yml --prune"
    )


print(f"Repo root: {REPO_ROOT}")
print(f"USE_GPU: {USE_GPU}")
print(f"nvidia_root exists: {nvidia_root.exists()} at {nvidia_root}")
print(f"ptxas found: {ptxas_path.exists()} at {ptxas_path if ptxas_path.exists() else 'N/A'}")
print(f"libnvrtc candidates: {[str(p) for p in nvrtc_candidates[:3]]}")
print(f"XLA_FLAGS: {os.environ.get('XLA_FLAGS', '')}")

2026-03-01 14:57:11.244770: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-01 14:57:11.244837: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-01 14:57:11.244894: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-01 14:57:11.254812: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 14:57:12.793537: I tensorflow/compiler/

TF version: 2.14.1
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Repo root: /home/dellio/github/har-mcu
USE_GPU: True


In [2]:
CONFIG_PATH = REPO_ROOT / "configs/default.yaml"
cfg_default = load_yaml(CONFIG_PATH)

if not USE_GPU:
    cfg_default = copy.deepcopy(cfg_default)
    cfg_default.setdefault("env", {})["require_gpu"] = False

runtime_status = check_tensorflow_runtime(cfg_default)
display(pd.DataFrame([runtime_status]))

if "error" in runtime_status:
    raise RuntimeError(runtime_status["error"])

print("Loaded config:", CONFIG_PATH)



,tensorflow_ok,version_ok,gpu_ok,tensorflow_version,gpus
0,True,True,True,2.14.1,[/physical_device:GPU:0]


Loaded config: /home/dellio/github/har-mcu/configs/default.yaml


In [3]:
# Parameters
RUN_MODE = "quick"  # "quick" or "full"
WINDOW_SIZE = 100
PROTOCOLS = ["random_stratified", "user_holdout"]
RUN_QAT = True
FORCE_RETRAIN = False

print({
    "RUN_MODE": RUN_MODE,
    "WINDOW_SIZE": WINDOW_SIZE,
    "PROTOCOLS": PROTOCOLS,
    "RUN_QAT": RUN_QAT,
    "FORCE_RETRAIN": FORCE_RETRAIN,
})


{'RUN_MODE': 'quick', 'WINDOW_SIZE': 100, 'PROTOCOLS': ['random_stratified', 'user_holdout'], 'RUN_QAT': True, 'FORCE_RETRAIN': False}


In [4]:
def build_notebook_cfg(base_cfg, run_mode, window_size, protocols):
    cfg = copy.deepcopy(base_cfg)
    cfg["window_size_default"] = int(window_size)
    cfg["split_protocols"] = list(protocols)

    if run_mode not in {"quick", "full"}:
        raise ValueError("RUN_MODE must be 'quick' or 'full'")

    if run_mode == "quick":
        cfg.setdefault("smoke", {})["enabled"] = True
        cfg["train"]["epochs"] = 1
        cfg["smoke"]["max_windows_per_class"] = 200
        cfg["quant"]["ptq"]["representative_samples"] = 32
        cfg["quant"]["qat"]["epochs"] = 1
    else:
        cfg.setdefault("smoke", {})["enabled"] = False
        cfg["smoke"]["max_windows_per_class"] = None

    return cfg

cfg_effective = build_notebook_cfg(cfg_default, RUN_MODE, WINDOW_SIZE, PROTOCOLS)
set_global_seed(int(cfg_effective["seed"]))

preview = {
    "window_size_default": cfg_effective["window_size_default"],
    "split_protocols": cfg_effective["split_protocols"],
    "epochs": cfg_effective["train"]["epochs"],
    "ptq_representative_samples": cfg_effective["quant"]["ptq"]["representative_samples"],
    "smoke_enabled": cfg_effective["smoke"]["enabled"],
    "smoke_max_windows_per_class": cfg_effective["smoke"].get("max_windows_per_class"),
}
print(json.dumps(preview, indent=2))


{
  "window_size_default": 100,
  "split_protocols": [
    "random_stratified",
    "user_holdout"
  ],
  "epochs": 1,
  "ptq_representative_samples": 32,
  "smoke_enabled": true,
  "smoke_max_windows_per_class": 200
}


In [5]:
raw_df, sanity = load_wisdm_dataframe(cfg_effective)
clean_df, pre_stats = preprocess_zhou2025(raw_df, cfg_effective)

class_counts = raw_df["activity"].value_counts().rename("count").reset_index().rename(columns={"index": "activity"})

sanity_table = pd.DataFrame([
    {
        "rows_raw": int(len(raw_df)),
        "rows_after_preprocess": int(len(clean_df)),
        "missing_values": int(sanity["missing_values"]),
        "zero_timestamps": int(sanity["zero_timestamps"]),
        "unique_users": int(raw_df["user"].nunique()),
    }
])

display(sanity_table)
display(class_counts)

paper_readme_rows = 1098207
local_rows = int(len(raw_df))
if local_rows != paper_readme_rows:
    print(
        f"Note: local CSV rows ({local_rows}) differ from README/paper-stated rows ({paper_readme_rows}). "
        "Replication should be interpreted directionally."
    )


,rows_raw,rows_after_preprocess,missing_values,zero_timestamps,unique_users
0,1073623,1073623,0,0,36


,activity,count
0,Walking,417901
1,Jogging,324600
2,Upstairs,122598
3,Downstairs,100192
4,Sitting,59939
5,Standing,48393


Note: local CSV rows (1073623) differ from README/paper-stated rows (1098207). Replication should be interpreted directionally.


In [ ]:
dataset_cards = {}
split_rows = []
class_rows = []

for protocol in PROTOCOLS:
    out = build_dataset_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    card = load_json(out["artifacts"]["datacard"])
    dataset_cards[protocol] = card

    split_rows.append(
        {
            "protocol": protocol,
            "window_size": card["windowing"]["window_size"],
            "step": card["windowing"]["step"],
            "candidate_windows": card["windowing"]["candidate_windows"],
            "windows_final": card["windowing"]["windows_final"],
            "train_size": card["split"]["train_size"],
            "val_size": card["split"]["val_size"],
            "test_size": card["split"]["test_size"],
            "split_hash": card["split"]["split_hash"],
        }
    )

    for cls, counts in card["counts"].items():
        class_rows.append(
            {
                "protocol": protocol,
                "activity": cls,
                "train": counts["train"],
                "val": counts["val"],
                "test": counts["test"],
            }
        )

split_df = pd.DataFrame(split_rows)
class_df = pd.DataFrame(class_rows)

display(split_df)
display(class_df)


,protocol,window_size,step,candidate_windows,windows_final,train_size,val_size,test_size,split_hash
0,random_stratified,100,50,21421,1200,643,161,396,e8b3eefc294da8de
1,user_holdout,100,50,21421,1200,217,180,803,8f23a2e13f210a67


,protocol,activity,train,val,test
0,random_stratified,Downstairs,107,27,66
1,random_stratified,Jogging,107,27,66
2,random_stratified,Sitting,107,27,66
3,random_stratified,Standing,107,27,66
4,random_stratified,Upstairs,107,27,66
5,random_stratified,Walking,108,26,66
6,user_holdout,Downstairs,30,59,111
7,user_holdout,Jogging,0,0,200
8,user_holdout,Sitting,90,31,79
9,user_holdout,Standing,71,29,100


: 

In [ ]:
baseline_train_rows = []

for protocol in PROTOCOLS:
    ckpt_path = baseline_ckpt_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)
    hist_path = history_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)

    if ckpt_path.exists() and not FORCE_RETRAIN:
        train_out = {
            "checkpoint": str(ckpt_path),
            "history": str(hist_path) if hist_path.exists() else None,
            "epochs_ran": 0,
            "final_val_accuracy": None,
            "reused_checkpoint": True,
        }
    else:
        train_out = train_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
        train_out["reused_checkpoint"] = False

    train_out["protocol"] = protocol
    baseline_train_rows.append(train_out)

baseline_train_df = pd.DataFrame(baseline_train_rows)
display(baseline_train_df)


2026-03-01 14:57:16.269072: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-01 14:57:16.269164: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-01 14:57:16.269203: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-01 14:57:16.467875: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-01 14:57:16.467955: I tensorflow/compile

In [ ]:
baseline_eval_metrics = {}
baseline_eval_rows = []

for protocol in PROTOCOLS:
    out = evaluate_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    m = load_json(out["metrics_json"])
    baseline_eval_metrics[protocol] = m

    baseline_eval_rows.append(
        {
            "protocol": protocol,
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "confusion_plot": m["confusion_plot"],
            "metrics_json": out["metrics_json"],
            "report_md": out["report_md"],
        }
    )

baseline_eval_df = pd.DataFrame(baseline_eval_rows)
display(baseline_eval_df)


In [ ]:
ptq_export_metrics = {}
ptq_eval_metrics = {}
ptq_rows = []

for protocol in PROTOCOLS:
    export_out = quantize_ptq_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    export_json = load_json(export_out["report_json"])

    eval_out = evaluate_tflite(
        cfg_effective,
        export_out["tflite_model"],
        WINDOW_SIZE,
        protocol,
        tag="ptq_nb",
    )
    eval_json = load_json(eval_out["metrics_json"])

    ptq_export_metrics[protocol] = export_json
    ptq_eval_metrics[protocol] = eval_json

    ptq_rows.append(
        {
            "protocol": protocol,
            "accuracy": eval_json["accuracy"],
            "macro_f1": eval_json["macro_f1"],
            "model_size_kb": eval_json["model_size_kb"],
            "input_dtype": export_json["input_dtype"],
            "output_dtype": export_json["output_dtype"],
            "tflite_model": export_out["tflite_model"],
            "eval_report_md": eval_out["report_md"],
        }
    )

ptq_df = pd.DataFrame(ptq_rows)
display(ptq_df)


In [ ]:
qat_export_metrics = {}
qat_eval_metrics = {}
qat_rows = []

if RUN_QAT:
    for protocol in PROTOCOLS:
        try:
            qat_out = qat_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
            export_json = load_json(qat_out["report_json"])
            qat_export_metrics[protocol] = export_json

            eval_out = evaluate_tflite(
                cfg_effective,
                qat_out["tflite"],
                WINDOW_SIZE,
                protocol,
                tag="qat_nb",
            )
            eval_json = load_json(eval_out["metrics_json"])
            qat_eval_metrics[protocol] = eval_json

            deployability = (
                "full-int8-deployable" if export_json.get("deployable_full_int8", False)
                else "non-deployable-fallback"
            )
            notes = " | ".join(export_json.get("notes", []))

            qat_rows.append(
                {
                    "protocol": protocol,
                    "accuracy": eval_json["accuracy"],
                    "macro_f1": eval_json["macro_f1"],
                    "model_size_kb": eval_json["model_size_kb"],
                    "deployability": deployability,
                    "input_dtype": export_json.get("input_dtype"),
                    "output_dtype": export_json.get("output_dtype"),
                    "notes": notes,
                    "eval_report_md": eval_out["report_md"],
                }
            )
        except Exception as exc:
            qat_rows.append(
                {
                    "protocol": protocol,
                    "accuracy": None,
                    "macro_f1": None,
                    "model_size_kb": None,
                    "deployability": "error",
                    "input_dtype": None,
                    "output_dtype": None,
                    "notes": str(exc),
                    "eval_report_md": None,
                }
            )
else:
    print("RUN_QAT=False -> skipping QAT section")

qat_df = pd.DataFrame(qat_rows)
display(qat_df)


In [ ]:
TARGET_BASELINE_ACC = 0.9824
TARGET_PTQ_ACC = 0.9709
TARGET_PTQ_SIZE_KB = 136.51

primary_protocol = "random_stratified"
if primary_protocol not in baseline_eval_metrics:
    raise ValueError("random_stratified results missing; cannot run verdict checks")
if primary_protocol not in ptq_eval_metrics:
    raise ValueError("PTQ results missing for random_stratified")

baseline_acc_random = float(baseline_eval_metrics[primary_protocol]["accuracy"])
ptq_acc_random = float(ptq_eval_metrics[primary_protocol]["accuracy"])
ptq_size_random = float(ptq_eval_metrics[primary_protocol]["model_size_kb"])

ptq_input_dtype = str(ptq_export_metrics[primary_protocol]["input_dtype"]).lower()
ptq_output_dtype = str(ptq_export_metrics[primary_protocol]["output_dtype"]).lower()

checks = [
    {
        "check": "Baseline accuracy close to paper target",
        "rule": "abs(baseline_acc - 0.9824) <= 0.02",
        "value": baseline_acc_random,
        "status": "PASS" if abs(baseline_acc_random - TARGET_BASELINE_ACC) <= 0.02 else "WARN",
    },
    {
        "check": "PTQ accuracy close to paper target",
        "rule": "abs(ptq_acc - 0.9709) <= 0.03",
        "value": ptq_acc_random,
        "status": "PASS" if abs(ptq_acc_random - TARGET_PTQ_ACC) <= 0.03 else "WARN",
    },
    {
        "check": "PTQ drop bounded vs baseline",
        "rule": "baseline-0.03 <= ptq <= baseline+0.005",
        "value": ptq_acc_random,
        "status": "PASS" if (baseline_acc_random - 0.03) <= ptq_acc_random <= (baseline_acc_random + 0.005) else "WARN",
    },
    {
        "check": "PTQ I/O dtype is int8",
        "rule": "input/output contain int8",
        "value": f"in={ptq_input_dtype}, out={ptq_output_dtype}",
        "status": "PASS" if ("int8" in ptq_input_dtype and "int8" in ptq_output_dtype) else "WARN",
    },
    {
        "check": "PTQ size in informational band",
        "rule": "80KB <= size <= 220KB",
        "value": ptq_size_random,
        "status": "PASS" if 80.0 <= ptq_size_random <= 220.0 else "WARN",
    },
]

verdict_df = pd.DataFrame(checks)
display(verdict_df)


In [ ]:
summary_rows = []

for protocol in PROTOCOLS:
    if protocol in baseline_eval_metrics:
        m = baseline_eval_metrics[protocol]
        summary_rows.append(
            {
                "protocol": protocol,
                "model": "baseline",
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
                "model_size_kb": None,
                "deployability": "n/a",
            }
        )

    if protocol in ptq_eval_metrics:
        m = ptq_eval_metrics[protocol]
        summary_rows.append(
            {
                "protocol": protocol,
                "model": "ptq",
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
                "model_size_kb": m["model_size_kb"],
                "deployability": "full-int8",
            }
        )

    if protocol in qat_eval_metrics:
        m = qat_eval_metrics[protocol]
        deployability = "full-int8-deployable" if qat_export_metrics.get(protocol, {}).get("deployable_full_int8", False) else "non-deployable-fallback"
        summary_rows.append(
            {
                "protocol": protocol,
                "model": "qat",
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
                "model_size_kb": m["model_size_kb"],
                "deployability": deployability,
            }
        )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

reports_dir = Path(cfg_effective["paths"]["reports_dir"])
reports_dir.mkdir(parents=True, exist_ok=True)

summary_csv_path = reports_dir / f"notebook_replication_summary_T{WINDOW_SIZE}.csv"
summary_md_path = reports_dir / f"notebook_replication_summary_T{WINDOW_SIZE}.md"

summary_df.to_csv(summary_csv_path, index=False)

with summary_md_path.open("w", encoding="utf-8") as f:
    f.write(f"# Notebook Replication Summary (T={WINDOW_SIZE})\n\n")
    f.write(f"- Run mode: `{RUN_MODE}`\n")
    f.write(f"- Protocols: {PROTOCOLS}\n\n")
    f.write("## Summary table\n\n")
    f.write("```\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n```\n\n")
    f.write("## Verdict table\n\n")
    f.write("```\n")
    f.write(verdict_df.to_string(index=False))
    f.write("\n```\n")

print("Saved:")
print("-", summary_csv_path)
print("-", summary_md_path)


In [ ]:
repro_rows = []
DRIFT_TOL = 1e-9

for protocol in PROTOCOLS:
    split_path = split_npz_path(cfg_effective["paths"]["processed_dir"], WINDOW_SIZE, protocol)
    split_hash = None
    if split_path.exists():
        split_npz = np.load(split_path, allow_pickle=True)
        raw_hash = split_npz["split_hash"]
        split_hash = raw_hash.item() if hasattr(raw_hash, "item") else str(raw_hash)

    repeat_out = evaluate_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    repeat_metrics = load_json(repeat_out["metrics_json"])

    old_acc = float(baseline_eval_metrics[protocol]["accuracy"])
    new_acc = float(repeat_metrics["accuracy"])
    drift = abs(new_acc - old_acc)

    repro_rows.append(
        {
            "protocol": protocol,
            "split_hash": split_hash,
            "baseline_acc_first": old_acc,
            "baseline_acc_repeat": new_acc,
            "abs_drift": drift,
            "status": "PASS" if drift <= DRIFT_TOL else "WARN",
        }
    )

repro_df = pd.DataFrame(repro_rows)
display(repro_df)


## Interpretation Notes and Limitations

- This notebook is designed to be **paper-faithful directionally**, while reusing the repository's deterministic script pipeline.
- Differences from paper values can come from dataset version differences, split realizations, and framework/kernel variations.
- QAT results are always reported. If full-int8 export is unsupported for LSTM in the active stack, results are retained and marked as non-deployable fallback.

## Suggested next steps
- Run `RUN_MODE = "full"` for final replication logs.
- Run window-size sweep scripts for broader comparison.
- Move successful PTQ artifacts into deployment profiling.
